# Challenge de la Sesión 05 — Evaluación de modelos generativos

*Desarrollo y Evaluación de Modelos Propios · Universidad del Rosario · Rosario GSB*

[![Abrir en Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/05-evaluating-generative-models/challenge.ipynb)
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/juccaicedoac03/Desarrollo-Evaluacion-Modelos/blob/main/sessions/05-evaluating-generative-models/challenge.ipynb)

> **Modalidad:** individual · **Tiempo:** 40 minutos en clase + entrega hasta las **23:59 de hoy** (hora de Colombia).
>
> **IA permitida con registro:** puedes usar asistentes de IA, pero debes declararlos en el registro del final. Sin registro, el componente de reflexión y registro de IA vale 0.0.
>
> **Micro-sustentaciones:** durante el bloque del challenge se eligen estudiantes al azar para explicar su notebook en 2–3 minutos, sin ayudas. Si no puedes explicar tu entrega, la nota de este challenge se limita a 3.0.
>
> **Rúbrica:** Ejecución técnica **40%** · Análisis e interpretación **40%** · Reflexión y registro de IA **20%**.
>
> **Entrega:** ejecuta todo en orden, luego `Archivo → Descargar notebook` (en Kaggle: *File → Download notebook*) y súbelo a **e-Aulas** como **`S05_<codigo>.ipynb`**.

**Qué vas a hacer**

| Tarea | Qué | Componente | Tiempo sugerido |
|---|---|---|---|
| 1 | Métricas sobre *tus* noticias, *tus* traducciones y *tu* k | Ejecución técnica | 15' |
| 2 | Un caso en el que la métrica y tu juicio no coinciden | Análisis e interpretación | 10' |
| 3 | **Explica y decide:** plan de evaluación para tu producto | Análisis e interpretación | 10' |
| 4 | **Crítica a la IA:** tres errores sobre métricas y validación | Análisis e interpretación | 5' |

**En Kaggle:** *Settings → Internet on*. No necesitas GPU: todo corre en CPU en pocos minutos.
Escribe tu código estudiantil en la primera celda **antes** de ejecutar nada más.

In [ ]:
# ✏️ Escribe tu código estudiantil (como aparece en e-Aulas) y tu nombre completo.
STUDENT_ID = ""
STUDENT_NAME = ""

# --- Configuración del curso (no editar) ---
import os, sys, random, subprocess, importlib, hashlib, json
SESSION = "S05"
FAST_DEV_RUN = os.environ.get("COURSE_FAST_DEV_RUN") == "1"
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
IN_COLAB = "google.colab" in sys.modules
if FAST_DEV_RUN and not STUDENT_ID.strip():
    STUDENT_ID, STUDENT_NAME = "000000", "Prueba automática"
assert STUDENT_ID.strip(), "⚠️ Escribe tu STUDENT_ID en la primera línea de esta celda y vuelve a ejecutarla."

def ensure(package, import_name=None):
    """Instala un paquete solo si falta."""
    try:
        importlib.import_module(import_name or package)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

PERSONAL_SEED = int(hashlib.sha256(f"{SESSION}:{STUDENT_ID.strip()}".encode()).hexdigest()[:8], 16)
rng = random.Random(PERSONAL_SEED)

def pick(options):
    """Elige una opción de forma determinística a partir de tu código estudiantil."""
    return rng.choice(list(options))

import numpy as np
import torch
random.seed(PERSONAL_SEED); np.random.seed(PERSONAL_SEED % (2**32)); torch.manual_seed(PERSONAL_SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Sesión {SESSION} | Estudiante: {STUDENT_NAME} ({STUDENT_ID}) | Semilla personal: {PERSONAL_SEED} | Dispositivo: {DEVICE}")
# --- fin de la configuración ---

# Paquetes de esta sesión (Kaggle ya trae la mayoría)
ensure("sacrebleu")
ensure("rouge-score", "rouge_score")
ensure("sentence-transformers", "sentence_transformers")
ensure("sentencepiece")  # lo necesita el tokenizador de opus-mt (MarianMT)

In [ ]:
# Tu configuración personal: depende de tu STUDENT_ID. No la modifiques.
PRODUCTOS = [
    "Asistente que redacta respuestas a reclamos de clientes de NovaTel (telecomunicaciones)",
    "Resumidor de historias clínicas para el personal de la EPS Salud Andina",
    "Traductor ES→EN de fichas de producto para Café Alto Montaña (exportador de café)",
    "Clasificador de intención de los correos que recibe Banco Andino (10 categorías)",
    "Generador de resúmenes diarios de noticias financieras para asesores de Andes Bank",
    "Detector de solicitudes de crédito con posible fraude documental en Andes Bank",
]
VENTANA_NOTICIAS = 200   # las primeras 200 noticias de validación de CNN/DailyMail

CONFIG = {
    "sample_ids": sorted(rng.sample(range(VENTANA_NOTICIAS), 8)),   # posiciones dentro de la ventana
    "mt_ids": sorted(rng.sample(range(100, 93000), 20)),            # filas de opus_books (en-es)
    "k": pick([3, 5, 10]),                                          # folds de la validación cruzada
    "product": pick(PRODUCTOS),                                     # producto de la Tarea 3
}
print("🎯 Tu configuración personal")
for clave, valor in CONFIG.items():
    print(f"  · {clave}: {valor}")

**Qué significa tu configuración**

- `sample_ids`: las 8 noticias de `abisee/cnn_dailymail` (validación) que vas a resumir. Son posiciones dentro de una ventana fija de 200 noticias que se leen en *streaming*.
- `mt_ids`: las 20 oraciones de `Helsinki-NLP/opus_books` (inglés → español) que vas a traducir. Si una fila es un título o una línea muy corta, se usa la siguiente fila válida.
- `k`: el número de *folds* de tu validación cruzada estratificada.
- `product`: el producto para el que diseñarás un plan de evaluación en la Tarea 3.

La siguiente celda descarga tus datos. Tarda alrededor de un minuto.

In [ ]:
import gc, re, time, warnings
import pandas as pd
from IPython.display import display
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
warnings.filterwarnings("ignore", message=".*sacremoses.*")
pd.set_option("display.max_colwidth", 200)

# Tamaños (FAST_DEV_RUN solo lo usa la prueba automática del curso: mismo código, menos datos)
N_RES = 3 if FAST_DEV_RUN else len(CONFIG["sample_ids"])
N_TRAD = 5 if FAST_DEV_RUN else len(CONFIG["mt_ids"])
MAX_TOKENS_RESUMEN = 32 if FAST_DEV_RUN else 64
MAX_TOKENS_TRAD = 64 if FAST_DEV_RUN else 128
NUM_BEAMS = 1 if FAST_DEV_RUN else 4
N_POR_INTENT = 40 if FAST_DEV_RUN else None   # None = todos los ejemplos de entrenamiento

# 1) Tus noticias: recorremos la ventana en streaming y guardamos solo tus posiciones
mis_pos = CONFIG["sample_ids"][:N_RES]
stream = load_dataset("abisee/cnn_dailymail", "3.0.0", split="validation", streaming=True)
noticias = []
for pos, ejemplo in enumerate(stream):
    if pos in mis_pos:
        noticias.append({"pos": pos, "id": ejemplo["id"], "articulo": ejemplo["article"],
                         "referencia": ejemplo["highlights"]})
    if pos >= max(mis_pos):
        break

# 2) Tus pares de traducción (se saltan títulos y líneas muy cortas o muy largas)
books = load_dataset("Helsinki-NLP/opus_books", "en-es", split="train")


def par_limpio(par, min_palabras=5, max_palabras=40):
    en, es = par["en"].strip(), par["es"].strip()
    return (min_palabras <= len(en.split()) <= max_palabras and len(es.split()) >= min_palabras
            and not en.isupper() and not en.lower().startswith(("source:", "chapter")))


def fila_valida(i, usadas):
    """La primera fila limpia a partir de i (determinística)."""
    while True:
        if i >= len(books):
            i = 100
        if i not in usadas and par_limpio(books[i]["translation"]):
            return i
        i += 1


filas_mt = []
for i in CONFIG["mt_ids"][:N_TRAD]:
    filas_mt.append(fila_valida(i, set(filas_mt)))
fuentes = [books[i]["translation"]["en"] for i in filas_mt]
referencias = [books[i]["translation"]["es"] for i in filas_mt]

print(f"✅ {len(noticias)} noticias (posiciones {[n['pos'] for n in noticias]}) · {len(fuentes)} pares de traducción")

## Tarea 1 — Métricas sobre tus muestras (Ejecución técnica)

Ejecuta las tres partes. Las celdas ya funcionan con valores razonables; donde veas ✏️ puedes
ajustar algo, pero **no cambies `CONFIG`** y reporta cualquier cambio que hagas.

- **1a · Resúmenes.** `google/flan-t5-small` resume tus noticias. Lo comparamos con la línea base
  **lead-3** (las tres primeras oraciones) usando ROUGE-1, ROUGE-2 y ROUGE-L (F1) y la similitud
  coseno de `sentence-transformers/all-MiniLM-L6-v2` contra los *highlights* de referencia.
- **1b · Traducciones.** `Helsinki-NLP/opus-mt-en-es` traduce tus oraciones; calculamos **BLEU** y
  **chrF** de corpus con `sacrebleu` y también por oración.
- **1c · Validación.** Validación cruzada estratificada con **tu k** de TF-IDF + regresión logística
  sobre 10 intenciones de Banking77 y, al final, una sola evaluación en el conjunto de prueba.

In [ ]:
# 1a · Resúmenes: modelo vs. línea base lead-3
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

tok_res = AutoTokenizer.from_pretrained("google/flan-t5-small")
modelo_res = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to(DEVICE).eval()

# ✏️ TU CÓDIGO AQUÍ (opcional): puedes probar otra instrucción, pero reporta cuál usaste.
PREFIJO = "summarize: "


def resumir(articulo):
    entradas = tok_res(PREFIJO + articulo, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)
    with torch.no_grad():
        salida = modelo_res.generate(**entradas, max_new_tokens=MAX_TOKENS_RESUMEN, num_beams=NUM_BEAMS)
    return tok_res.decode(salida[0], skip_special_tokens=True)


def lead3(articulo):
    oraciones = re.split(r"(?<=[.!?])\s+", articulo.strip())
    return " ".join(oraciones[:3])


t0 = time.time()
for n in noticias:
    n["modelo"] = resumir(n["articulo"])
    n["lead3"] = lead3(n["articulo"])
print(f"Resúmenes generados en {time.time() - t0:.0f} s")

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)


def coseno(textos_a, textos_b):
    a = embedder.encode(textos_a, convert_to_tensor=True, normalize_embeddings=True)
    b = embedder.encode(textos_b, convert_to_tensor=True, normalize_embeddings=True)
    return util.cos_sim(a, b).diagonal().cpu().numpy()


filas = []
for sistema in ["modelo", "lead3"]:
    cos = coseno([n[sistema] for n in noticias], [n["referencia"] for n in noticias])
    for n, c in zip(noticias, cos):
        s = scorer.score(n["referencia"], n[sistema])      # score(referencia, candidato)
        filas.append({"pos": n["pos"], "sistema": sistema, "rouge1": s["rouge1"].fmeasure,
                      "rouge2": s["rouge2"].fmeasure, "rougeL": s["rougeL"].fmeasure,
                      "coseno": float(c), "palabras": len(n[sistema].split())})
tabla_res = pd.DataFrame(filas)
promedios_res = tabla_res.groupby("sistema")[["rouge1", "rouge2", "rougeL", "coseno", "palabras"]].mean()
display(promedios_res.round(4))

del modelo_res
gc.collect()

In [ ]:
# 1b · Traducciones EN→ES: BLEU y chrF
import sacrebleu

tok_mt = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
modelo_mt = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es").to(DEVICE).eval()


def traducir(textos, lote=8):
    salida = []
    for i in range(0, len(textos), lote):
        entradas = tok_mt(textos[i:i + lote], return_tensors="pt", padding=True,
                          truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            gen = modelo_mt.generate(**entradas, max_new_tokens=MAX_TOKENS_TRAD, num_beams=NUM_BEAMS)
        salida += tok_mt.batch_decode(gen, skip_special_tokens=True)
    return salida


hipotesis = traducir(fuentes)
bleu = sacrebleu.corpus_bleu(hipotesis, [referencias])
chrf = sacrebleu.corpus_chrf(hipotesis, [referencias])
print(bleu)
print(chrf)

tabla_mt = pd.DataFrame({
    "fila": filas_mt,
    "fuente (EN)": fuentes,
    "modelo (ES)": hipotesis,
    "referencia (ES)": referencias,
    "BLEU": [sacrebleu.sentence_bleu(h, [r]).score for h, r in zip(hipotesis, referencias)],
    "chrF": [sacrebleu.sentence_chrf(h, [r]).score for h, r in zip(hipotesis, referencias)],
})
display(tabla_mt.sort_values("BLEU").round(1))

del modelo_mt
gc.collect()

In [ ]:
# 1c · Validación cruzada estratificada con tu k (Banking77, 10 intenciones)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score

INTENTS = ["card_arrival", "card_delivery_estimate", "lost_or_stolen_card", "card_payment_not_recognised",
           "pending_transfer", "transfer_not_received_by_recipient", "exchange_rate",
           "card_payment_wrong_exchange_rate", "pending_top_up", "top_up_failed"]
banking = load_dataset("legacy-datasets/banking77")
nombres = banking["train"].features["label"].names
ids_intents = {nombres.index(i) for i in INTENTS}


def a_tabla(split):
    df = split.to_pandas()
    df = df[df["label"].isin(ids_intents)].copy()
    df["intent"] = df["label"].map(lambda i: nombres[i])
    return df[["text", "intent"]].reset_index(drop=True)


train_df, test_df = a_tabla(banking["train"]), a_tabla(banking["test"])
if N_POR_INTENT:
    train_df = train_df.groupby("intent", group_keys=False).sample(N_POR_INTENT, random_state=PERSONAL_SEED % (2**32))


def nuevo_modelo():
    return make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True), LogisticRegression(max_iter=2000))


k = CONFIG["k"]
cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=PERSONAL_SEED % (2**32))
cv_f1 = cross_val_score(nuevo_modelo(), train_df["text"], train_df["intent"], cv=cv, scoring="f1_macro")
print(f"F1 macro por fold (k = {k}):", np.round(cv_f1, 3))
print(f"Validación cruzada: {cv_f1.mean():.4f} ± {cv_f1.std(ddof=1):.4f} (media ± desviación estándar)")

# Una sola evaluación en el conjunto de prueba, con el modelo entrenado en todo el entrenamiento
modelo_final = nuevo_modelo().fit(train_df["text"], train_df["intent"])
test_f1 = f1_score(test_df["intent"], modelo_final.predict(test_df["text"]), average="macro")
print(f"F1 macro en prueba (una sola vez): {test_f1:.4f} · entrenamiento: {len(train_df)} · prueba: {len(test_df)}")

**Tu respuesta:** (3–4 líneas) ¿Tu modelo de resúmenes supera a lead-3 en ROUGE-L y en coseno? ¿Por
cuánto? ¿Qué diferencia ves entre tu BLEU y tu chrF, y por qué crees que ocurre? ¿El F1 de prueba
cae dentro del rango media ± desviación de tu validación cruzada?

## Tarea 2 — Cuando la métrica y tú no están de acuerdo (Análisis e interpretación)

Las métricas automáticas son baratas, pero no leen como una persona. En esta tarea **lees** tus
resultados y buscas un caso en el que la métrica diga una cosa y tu juicio otra.

1. Ejecuta la celda siguiente: muestra cada resumen del modelo con su referencia y sus métricas, y
   las traducciones con el BLEU más bajo y más alto.
2. Califica cada resumen del modelo de **1** (inútil o con errores) a **5** (fiel y completo) en
   `MIS_NOTAS`. Con al menos 3 notas se calcula la correlación de Spearman entre tus notas y las métricas.
3. Elige **un** caso de desacuerdo (un resumen o una traducción) y descríbelo en `CASO_DESACUERDO`.

In [ ]:
for n in noticias:
    fila = tabla_res[(tabla_res.pos == n["pos"]) & (tabla_res.sistema == "modelo")].iloc[0]
    print(f"── pos {n['pos']} · ROUGE-L {fila.rougeL:.3f} · coseno {fila.coseno:.3f}")
    print("   REFERENCIA:", n["referencia"].replace("\n", " "))
    print("   MODELO    :", n["modelo"])

print("\n══ Traducciones con BLEU más bajo y más alto")
for _, fila in pd.concat([tabla_mt.nsmallest(2, "BLEU"), tabla_mt.nlargest(1, "BLEU")]).iterrows():
    print(f"── fila {fila['fila']} · BLEU {fila['BLEU']:.1f} · chrF {fila['chrF']:.1f}")
    print("   EN :", fila["fuente (EN)"])
    print("   MT :", fila["modelo (ES)"])
    print("   REF:", fila["referencia (ES)"])

In [ ]:
# ✏️ TU CÓDIGO AQUÍ: califica cada resumen del modelo de 1 a 5 usando su posición como clave.
# Ejemplo: MIS_NOTAS = {17: 2, 45: 4, 101: 1}
MIS_NOTAS = {}

# ✏️ TU CÓDIGO AQUÍ: describe tu caso de desacuerdo.
# tipo: "resumen" (usa la pos) o "traduccion" (usa la fila); metrica: "rougeL", "coseno", "BLEU" o "chrF"
CASO_DESACUERDO = {"tipo": "resumen", "id": None, "metrica": "rougeL", "valor": None, "tu_nota_1a5": None}

rho_rougeL = rho_coseno = None
modelo_df = tabla_res[tabla_res.sistema == "modelo"].set_index("pos")
comunes = [p for p in modelo_df.index if p in MIS_NOTAS]
if len(comunes) >= 3:
    from scipy.stats import spearmanr
    notas = [MIS_NOTAS[p] for p in comunes]
    rho_rougeL = spearmanr(notas, modelo_df.loc[comunes, "rougeL"])[0]
    rho_coseno = spearmanr(notas, modelo_df.loc[comunes, "coseno"])[0]
    rho_rougeL = None if np.isnan(rho_rougeL) else float(rho_rougeL)
    rho_coseno = None if np.isnan(rho_coseno) else float(rho_coseno)
    print(f"Spearman(tus notas, ROUGE-L) = {rho_rougeL} · Spearman(tus notas, coseno) = {rho_coseno}")
else:
    print(f"Llevas {len(comunes)} notas en MIS_NOTAS: califica al menos 3 resúmenes para calcular la correlación.")
print("Tu caso de desacuerdo:", CASO_DESACUERDO)

**Tu respuesta:** (6–8 líneas)
1. ¿Cuál es tu caso de desacuerdo? Cita el texto (resumido), el valor de la métrica y tu nota.
2. ¿Por qué la métrica no coincide con tu juicio? Considera: paráfrasis, referencia libre o mal
   alineada, error factual con mucho solapamiento de palabras, longitud, o que el modelo haya visto datos parecidos.
3. ¿Qué otra forma de evaluación habría detectado el problema (otra métrica, evaluación humana, LLM como juez con qué precauciones)?
4. Si calificaste al menos 3 resúmenes: ¿qué dice tu correlación de Spearman? ¿Por qué hay que tomarla con cautela con tan pocos ejemplos?

## Tarea 3 — Explica y decide: plan de evaluación para tu producto (Análisis e interpretación)

Una gerente te pide un plan para decidir si el producto de tu configuración sale a producción.
Completa la tabla con decisiones concretas y **cita al menos 3 números tuyos** de las Tareas 1 y 2
(por ejemplo, la brecha entre el modelo y lead-3, la diferencia entre BLEU y chrF o la desviación
estándar de tu validación cruzada) para justificar tus elecciones.

In [ ]:
print("🧩 Tu producto:", CONFIG["product"])
print(f"   Tus números de referencia: ROUGE-L modelo {promedios_res.loc['modelo', 'rougeL']:.3f} vs lead-3 "
      f"{promedios_res.loc['lead3', 'rougeL']:.3f} · BLEU {bleu.score:.1f} · chrF {chrf.score:.1f} · "
      f"CV (k={k}) {cv_f1.mean():.3f} ± {cv_f1.std(ddof=1):.3f} · prueba {test_f1:.3f}")

**Tu respuesta:**

| Elemento del plan | Tu decisión | Justificación (con tus números) |
|---|---|---|
| Métricas automáticas (al menos 2) y línea base | | |
| Evaluación humana: quién, cuántos ejemplos, con qué criterios | | |
| ¿LLM como juez? Si lo usas, cómo controlas sus sesgos | | |
| KPI de negocio que debería moverse | | |
| Umbrales para salir a producción (*go / no-go*) | | |
| Estrategia de validación: particiones, k-fold, conjunto de prueba, riesgos de fuga | | |
| Monitoreo después del lanzamiento | | |

**Decisión (2–3 líneas):** ¿qué necesitas ver para aprobar el lanzamiento y qué harías si el modelo no supera la línea base?

## Tarea 4 — Crítica a la IA (Análisis e interpretación)

Un compañero le pidió a un asistente de IA un plan de evaluación y pegó esta respuesta en el
documento del equipo. **El texto tiene 3 errores conceptuales.** Encuéntralos, explica por qué
son errores y corrígelos. Puedes usar la celda de código como evidencia.

> *"Para evaluar el traductor y el asistente propongo lo siguiente. (1) Usaremos BLEU porque mide el
> **recall**: qué proporción de las palabras de la traducción de referencia aparece en la salida del
> modelo; así sabremos si se omite información. (2) Lo complementaremos con chrF, que compara n-gramas
> de caracteres y por eso tolera mejor las variaciones morfológicas del español. (3) Para escoger el
> modelo de lenguaje base compararemos perplejidades sobre nuestros textos: el modelo A obtuvo 18 y el
> modelo B 27; aunque usan tokenizadores distintos, una perplejidad más baja siempre indica un mejor
> modelo, así que elegiremos A. (4) Para el clasificador de intenciones haremos validación cruzada
> estratificada con 5 folds y con ella ajustaremos los hiperparámetros; como la validación cruzada ya
> evalúa con todos los datos, no hace falta reservar un conjunto de prueba y el promedio de los folds
> será el desempeño que reportaremos a la gerencia. (5) Además, revisaremos a mano una muestra de
> errores del clasificador para agruparlos por categorías."*

In [ ]:
# ✏️ TU CÓDIGO AQUÍ (opcional): evidencia para tu crítica. Por defecto, dos experimentos rápidos.

# Evidencia A: una salida muy corta con palabras correctas. ¿Qué pasa con las precisiones y con BLEU?
corta = " ".join(hipotesis[0].split()[:4])
b = sacrebleu.sentence_bleu(corta, [referencias[0]])
print(f"Salida corta: '{corta}'")
print(f"  precisiones n-gramas (%): {[round(p, 1) for p in b.precisions]} · BP = {b.bp:.3f} · BLEU = {b.score:.1f}")

# Evidencia B: el mismo texto en español, cortado por dos tokenizadores distintos
texto_es = referencias[0]
for nombre, tok in [("flan-t5-small", tok_res), ("opus-mt-en-es (tokenizador de entrada)", tok_mt)]:
    n_tokens = len(tok(texto_es)["input_ids"])
    print(f"  {nombre:>14}: {n_tokens} tokens para {len(texto_es.split())} palabras")

**Tu respuesta:**

| # | Afirmación de la IA (cita corta) | ¿Por qué es incorrecta? | Corrección |
|---|---|---|---|
| 1 | | | |
| 2 | | | |
| 3 | | | |

In [ ]:
# Resultados principales (no edites los nombres de las claves)
RESULTS = {
    "n_resumenes": len(noticias),
    "n_traducciones": len(hipotesis),
    "rouge1_modelo": round(float(promedios_res.loc["modelo", "rouge1"]), 4),
    "rouge2_modelo": round(float(promedios_res.loc["modelo", "rouge2"]), 4),
    "rougeL_modelo": round(float(promedios_res.loc["modelo", "rougeL"]), 4),
    "rougeL_lead3": round(float(promedios_res.loc["lead3", "rougeL"]), 4),
    "coseno_modelo": round(float(promedios_res.loc["modelo", "coseno"]), 4),
    "coseno_lead3": round(float(promedios_res.loc["lead3", "coseno"]), 4),
    "bleu": round(float(bleu.score), 4),
    "chrf": round(float(chrf.score), 4),
    "k": k,
    "cv_f1_macro_media": round(float(cv_f1.mean()), 4),
    "cv_f1_macro_std": round(float(cv_f1.std(ddof=1)), 4),
    "test_f1_macro": round(float(test_f1), 4),
    "n_notas_humanas": len(comunes),
    "spearman_notas_rougeL": None if rho_rougeL is None else round(rho_rougeL, 4),
    "spearman_notas_coseno": None if rho_coseno is None else round(rho_coseno, 4),
    "caso_desacuerdo": CASO_DESACUERDO,
}
print(json.dumps(RESULTS, indent=2, ensure_ascii=False))

## 🪞 Reflexión y registro de uso de IA (20%)

### Reflexión (responde en 3–5 líneas cada una)
1. ¿Qué aprendiste hoy que puedas aplicar en tu trabajo u organización?
2. ¿Qué fue lo más difícil y cómo lo resolviste?
3. Autoevaluación: ¿qué nota (0.0–5.0) te pondrías en este challenge y por qué?

### 🤖 Registro de uso de IA (obligatorio)
| # | Herramienta (ChatGPT, Claude, Codex, Copilot…) | ¿Para qué la usaste? | Prompt principal (resumido) | ¿Qué verificaste o corregiste de su respuesta? |
|---|---|---|---|---|
| 1 |  |  |  |  |
| 2 |  |  |  |  |

> Si no usaste asistentes de IA, escribe: *"No usé asistentes de IA en este challenge."* Recuerda: puedes usar IA, pero **eres responsable de todo lo que entregas** y podrías ser seleccionado(a) para una micro-sustentación.

In [ ]:
# --- Huella de resultados (no editar) ---
def results_fingerprint(results):
    payload = json.dumps({"session": SESSION, "student": STUDENT_ID.strip(), "results": results},
                         sort_keys=True, default=str, ensure_ascii=False)
    return hashlib.sha256(payload.encode()).hexdigest()[:12]

print(json.dumps({"CONFIG": CONFIG, "RESULTS": RESULTS}, indent=2, ensure_ascii=False, default=str))
print("🔏 Huella de resultados:", results_fingerprint({"CONFIG": CONFIG, "RESULTS": RESULTS}))